In [1]:
!pip install -U datasets huggingface_hub pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's depende

In [1]:
import os
from datasets import Dataset, Features, Value, Image
from huggingface_hub import login

In [2]:
from google.colab import userdata
userdata.get('')
# 1) Login (paste your HF token)
login(token="")

In [4]:
# 2) Your image folder
IMG_DIR = "./iphone_imgs"  # change this
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

In [5]:
# 3) Write captions (you can edit these)
# Keep it short & consistent.
captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

In [6]:
assert len(image_files) == len(captions), "Images and captions count must match!"

instruction = "Describe this iPhone product image in one sentence."

In [7]:
rows = []
for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [8]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

ds = Dataset.from_list(rows, features=features)

# If you want to store messages as real objects, easiest is to NOT force Features for messages.
#ds = Dataset.from_list(rows)  # messages stored as nested objects

# 4) Make train/test splits (with only 5 images, keep test=1)
splits = ds.train_test_split(test_size=1, seed=3407)

# 5) Push to Hub
REPO_ID = "sunny199/iphone5_vlm_img"  # change this
splits["train"].push_to_hub(REPO_ID, split="train")
splits["test"].push_to_hub(REPO_ID, split="test")

print("Uploaded:", REPO_ID)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 10.7kB / 10.7kB            

README.md:   0%|          | 0.00/335 [00:00<?, ?B/s]

Uploaded: sunny199/iphone5_vlm_img


In [9]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceM4/ChartQA")

print(dataset)

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [10]:
from datasets import load_dataset
dataset = load_dataset("HuggingFaceM4/ChartQA")
print(dataset)
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7E319C7B2F00>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}
